In [1]:
import pandas as pd 

df = pd.read_csv('Results.csv',header=None)
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID','Country']

df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info()

C:\Users\varun\AppData\Local\Temp\ipykernel_24068\3368499431.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Results.csv',header=None)


(1033036, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 63.1+ MB


In [2]:
# Filter And Calculate Recency , Frequency , Monetary
rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity']>0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

# RFM Analysis
rfm = df.groupby('Customer_ID').agg(

    # Recency
    Recency=('InvoiceDate',
             lambda x: (snapshot_date - x.max()).days),

    # Frequency
    Frequency=('Invoice', 'nunique'),

    # Monetary
    Monetary=('Quantity',
              lambda x: (x * df.loc[x.index, 'Price']).sum())

).reset_index()

print(rfm)

      Customer_ID  Recency  Frequency  Monetary
0         12346.0      325         17    -51.74
1         12347.0        2          8   4921.53
2         12348.0       75          5   2019.40
3         12349.0       18          5   4404.54
4         12350.0      310          1    334.40
...           ...      ...        ...       ...
5937      18283.0        3         22   2664.90
5938      18284.0      429          2    436.68
5939      18285.0      660          1    427.00
5940      18286.0      476          3   1188.43
5941      18287.0       42          8   4177.89

[5942 rows x 4 columns]


In [3]:
# 1. Calculate RFM Scores

# Recency Score
rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)


# Frequency Score
rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Monetary Score
rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# 2. Customer Segmentation

def segment_customer(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'

    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost'


# 3. Apply Customer Segmentation

rfm['Segment'] = rfm.apply(segment_customer, axis=1)


# 4. Identify Churned Customers

rfm['IsChurned'] = rfm['Recency'] > 90


# 5. Display Final Results

print(rfm.head(20))

    Customer_ID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0       12346.0      325         17    -51.74        2        5        1   
1       12347.0        2          8   4921.53        5        4        5   
2       12348.0       75          5   2019.40        3        3        4   
3       12349.0       18          5   4404.54        5        3        5   
4       12350.0      310          1    334.40        2        1        2   
5       12351.0      375          1    300.93        2        1        2   
6       12352.0       36         13   1889.21        4        5        4   
7       12353.0      204          2    406.76        2        2        2   
8       12354.0      232          1   1079.40        2        1        3   
9       12355.0      214          2    947.61        2        2        3   
10      12356.0       22          6   6371.73        4        4        5   
11      12357.0       33          3  18287.66        4        2        5   
12      1235

In [4]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary',
         'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID',
    how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)
